# Kelompokkan gambar berdasarkan tanggal pengukuran sensor

Notebook ini memilih gambar `obsbot_photo_YYYYMMDD_HHMMSS_*.jpg` pada tanggal pengukuran sensor yang berada dalam rentang maksimal satu jam sebelum atau sesudah waktu sensor. Gambar dikelompokkan per tanggal sensor menggunakan hard link agar tidak menggandakan isi file, lalu dibuat CSV indeks.

In [1]:
from datetime import datetime
from pathlib import Path
import os
import re

import numpy as np
import pandas as pd

csv_file = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\Lab Analog K1-K16\gabungan_lab_analog.csv"
)
image_dir = Path(
    r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\Data Kultivasi Mikroalga K1-16\gambar_satu_per_sensor_unik\semua_gambar_jpg"
)
output_dir = image_dir.parent / "gambar_satu_per_sensor_unik_dipasangkan"
output_csv = csv_file.with_name("gabungan_lab_analog_gambar_unik.csv")

kolom_tanggal = "Tanggal"
kolom_waktu = "Waktu (jam:menit:detik)"

# Baca timestamp sensor
df_sensor = pd.read_csv(
    csv_file,
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)

kolom_tidak_ada = [
    kolom for kolom in (kolom_tanggal, kolom_waktu)
    if kolom not in df_sensor.columns
]
if kolom_tidak_ada:
    raise KeyError(f"Kolom sensor tidak ditemukan: {kolom_tidak_ada}")

tanggal = pd.to_datetime(
    df_sensor[kolom_tanggal].str.strip(),
    format="%Y-%m-%d",
    errors="coerce",
)
jam = df_sensor[kolom_waktu].astype(str).str.extract(
    r"(\d{1,2}:\d{2}(?::\d{2})?)",
    expand=False,
)

timestamp_sensor = pd.to_datetime(
    tanggal.dt.strftime("%Y-%m-%d") + " " + jam.fillna(""),
    format="mixed",
    errors="coerce",
)

if timestamp_sensor.isna().any():
    baris_invalid = (df_sensor.index[timestamp_sensor.isna()] + 2).tolist()
    raise ValueError(
        f"Timestamp sensor tidak valid pada baris CSV: {baris_invalid}"
    )

# Cari gambar bernama obsbot_photo_YYYYMMDD_HHMMSS_*.jpg
pola = re.compile(
    r"obsbot_photo_(\d{8})_(\d{6})_.+\.jpg$",
    re.IGNORECASE,
)

gambar = []

for path in image_dir.rglob("*"):
    if not path.is_file() or path.suffix.lower() != ".jpg":
        continue

    cocok = pola.search(path.name)
    if not cocok:
        continue

    try:
        timestamp_gambar = datetime.strptime(
            f"{cocok.group(1)} {cocok.group(2)}",
            "%Y%m%d %H%M%S",
        )
    except ValueError:
        continue

    gambar.append((pd.Timestamp(timestamp_gambar), path))

gambar.sort(key=lambda item: item[0])

if not gambar:
    raise FileNotFoundError(
        f"Tidak ditemukan gambar obsbot_photo yang valid di: {image_dir}"
    )

# Cari gambar dengan selisih waktu terkecil untuk setiap sensor.
# Jika selisihnya sama, gambar yang lebih awal dipilih.
waktu_gambar_ns = np.array(
    [timestamp.value for timestamp, _ in gambar],
    dtype=np.int64,
)
# Samakan satuan ke nanodetik; Pandas dapat menyimpan datetime dengan resolusi detik/mikrodetik.
waktu_sensor_ns = np.array([ts.value for ts in timestamp_sensor], dtype=np.int64)
posisi = np.searchsorted(waktu_gambar_ns, waktu_sensor_ns)

pilihan = []
gambar_terpakai = set()

for ts_sensor, target_ns, idx in zip(
    timestamp_sensor, waktu_sensor_ns, posisi
):
    kandidat = []

    if idx > 0:
        kandidat.append(idx - 1)
    if idx < len(gambar):
        kandidat.append(idx)

    idx_pilih = min(
        (i for i in range(len(gambar)) if i not in gambar_terpakai),
        key=lambda i: (abs(waktu_gambar_ns[i] - target_ns), waktu_gambar_ns[i]),
    )
    gambar_terpakai.add(idx_pilih)

    ts_gambar, path_gambar = gambar[idx_pilih]
    pilihan.append(
        (ts_sensor, ts_gambar, path_gambar, abs(ts_gambar - ts_sensor))
    )

# Buat folder per tanggal sensor dan kelompokkan gambar di dalamnya.
output_dir.mkdir(parents=True, exist_ok=True)

nama_gambar = []
timestamp_gambar_terpilih = []
selisih_waktu = []
path_hasil = []

for nomor, (ts_sensor, ts_gambar, path_gambar, delta) in enumerate(
    pilihan,
    start=1,
):
    tanggal_sensor = ts_sensor.strftime("%Y-%m-%d")
    folder_hari = output_dir / tanggal_sensor
    folder_hari.mkdir(parents=True, exist_ok=True)

    # Sertakan nama folder kultivasi dan nomor baris agar nama file unik.
    folder_asal = path_gambar.parent.parent.name
    tujuan = folder_hari / (
        f"{nomor:04d}_{folder_asal}_{path_gambar.name}"
    )

    # Hard link mengelompokkan file tanpa menggandakan isi gambar.
    if not tujuan.exists():
        os.link(path_gambar, tujuan)

    nama_gambar.append(path_gambar.name)
    timestamp_gambar_terpilih.append(
        ts_gambar.strftime("%Y-%m-%d %H:%M:%S")
    )
    selisih_waktu.append(str(delta))
    path_hasil.append(str(tujuan))

# Simpan tabel sensor beserta gambar yang dipilih.
hasil = df_sensor.copy()
hasil["Timestamp_Sensor"] = timestamp_sensor.dt.strftime(
    "%Y-%m-%d %H:%M:%S"
)
hasil["Nama_Gambar_Terdekat"] = nama_gambar
hasil["Timestamp_Gambar"] = timestamp_gambar_terpilih
hasil["Selisih_Dari_Sensor"] = selisih_waktu
hasil["Path_Gambar_Hasil"] = path_hasil

hasil.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"Baris sensor: {len(hasil)}")
print(f"Baris sensor yang mendapat gambar: {len(pilihan)}")
print(f"Jumlah file gambar yang diperiksa: {len(gambar)}")
print(f"Gambar unik yang dipilih: {len({item[2] for item in pilihan})}")
print(f"CSV hasil: {output_csv}")
print(f"Folder gambar terpilih: {output_dir}")

Baris sensor: 253
Baris sensor yang mendapat gambar: 253
Jumlah file gambar yang diperiksa: 253
Gambar unik yang dipilih: 253
CSV hasil: D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\Lab Analog K1-K16\gabungan_lab_analog_gambar_unik.csv
Folder gambar terpilih: D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\Data Kultivasi Mikroalga K1-16\gambar_satu_per_sensor_unik\gambar_satu_per_sensor_unik_dipasangkan
